<a href="https://colab.research.google.com/github/martinn308/AI-FOR-SCIENCE-AND-ENGINEERING-JKUAT/blob/Week-2/Week3_AHP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 Assignment — Dataset, Features, and ML Workflow
**Problem:** Predicting demand and fair pricing for affordable housing units across different areas of Kenya, connected to my work as a Data Analyst on the Kenya Affordable Housing Project (APH).

**Intended AI/ML outcome:** A model that can **predict** housing prices/demand for a given area based on factors such as income levels, population, and property characteristics, and help **classify** areas as high or low priority for new affordable housing projects.


## Data Needed for This Project

Ideally, the dataset would include, for each area/region in Kenya:
- Historical housing prices / sale prices
- Household income levels
- Population and population growth
- Number of bedrooms/rooms and house size
- Construction cost indicators
- Access to infrastructure (roads, water, electricity)
- Housing demand indicators (waiting lists, uptake rates)

 Kenya-specific, project-level housing data is not publicly available in ready-to-use form, so i used a **substitute public dataset ** with the same type of structure  from a public GitHub source. This lets me practice the same workflow I would apply once I get access to real Kenya Affordable housing project data.


## Importing the Dataset

Dataset used: **USA Housing dataset** (5000 rows, 7 columns) — a substitute dataset with the same structure as what I'd need for the Kenya housing project (area characteristics → house price).

Source: https://github.com/Shivang1179793/House-Price-Prediction


In [1]:
import pandas as pd

# Load the dataset (hosted on GitHub as a substitute for Kenya-specific housing data)
url = "https://raw.githubusercontent.com/Shivang1179793/House-Price-Prediction/main/housing.csv"
df = pd.read_csv(url)

print("Shape of dataset:", df.shape)
df.head()


Shape of dataset: (5000, 7)


,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701..."
1,79248.642455,6.002900,6.730821,3.09,40173.072174,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA..."
2,61287.067179,5.865890,8.512727,5.13,36882.159400,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482..."
3,63345.240046,7.188236,5.586729,3.26,34310.242831,1.260617e+06,USS Barnett\nFPO AP 44820
4,59982.197226,5.040555,7.839388,4.23,26354.109472,6.309435e+05,USNS Raymond\nFPO AE 09386


In [2]:
# Quick look at data types and basic stats
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Avg. Area Income              5000 non-null   float64
 1   Avg. Area House Age           5000 non-null   float64
 2   Avg. Area Number of Rooms     5000 non-null   float64
 3   Avg. Area Number of Bedrooms  5000 non-null   float64
 4   Area Population               5000 non-null   float64
 5   Price                         5000 non-null   float64
 6   Address                       5000 non-null   object 
dtypes: float64(6), object(1)
memory usage: 273.6+ KB


In [3]:
df.describe()


,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5.000000e+03
mean,68583.108984,5.977222,6.987792,3.981330,36163.516039,1.232073e+06
std,10657.991214,0.991456,1.005833,1.234137,9925.650114,3.531176e+05
min,17796.631190,2.644304,3.236194,2.000000,172.610686,1.593866e+04
25%,61480.562388,5.322283,6.299250,3.140000,29403.928702,9.975771e+05
50%,68804.286404,5.970429,7.002902,4.050000,36199.406689,1.232669e+06
75%,75783.338666,6.650808,7.665871,4.490000,42861.290769,1.471210e+06
max,107701.748378,9.519088,10.759588,6.500000,69621.713378,2.469066e+06


## Potential Inputs (Features) and Output (Target)

**Inputs (features):**
- `Avg. Area Income`
- `Avg. Area House Age`
- `Avg. Area Number of Rooms`
- `Avg. Area Number of Bedrooms`
- `Area Population`

(`Address` is dropped — it is just a text label, not a usable numeric/categorical feature in this form.)

**Output (target):**
- `Price` — the value the model will try to predict.

For the real Kenya housing project, the same idea applies: features would be things like area income, population, house size, and infrastructure access, and the target would be price (or a demand score).


In [5]:
features = [
    "Avg. Area Income",
    "Avg. Area House Age",
    "Avg. Area Number of Rooms",
    "Avg. Area Number of Bedrooms",
    "Area Population",
]
target = "Price"

X = df[features]
y = df[target]

print("Feature columns:", list(X.columns))
print("Target column:", target)
X.head()


Feature columns: ['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Avg. Area Number of Bedrooms', 'Area Population']
Target column: Price


,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population
0,79545.458574,5.682861,7.009188,4.09,23086.800503
1,79248.642455,6.002900,6.730821,3.09,40173.072174
2,61287.067179,5.865890,8.512727,5.13,36882.159400
3,63345.240046,7.188236,5.586729,3.26,34310.242831
4,59982.197226,5.040555,7.839388,4.23,26354.109472


## Proposed ML Workflow

```
 ┌─────────────────────┐
 │   1. Collect Data   │   (area income, population, house features, price)
 └──────────┬──────────┘
            │
 ┌──────────▼───────────┐
 │  2. Clean & Prepare  │   (handle missing values, drop text columns,
 │       Data           │    encode categories if needed)
 └──────────┬───────────┘
            │
 ┌──────────▼───────────┐
 │ 3. Split into Train /│   ( 80% train, 20% test)
 │        Test          │
 └──────────┬───────────┘
            │
 ┌──────────▼───────────┐
 │  4. Train ML Model   │   ( Linear Regression to start)
 └──────────┬───────────┘
            │
 ┌──────────▼───────────┐
 │  5. Evaluate Model   │   (compareS predicted vs actual price,
 │                      │    checks error metrics)
 └──────────┬───────────┘
            │
 ┌──────────▼───────────┐
 │6.Use Model to Predict│   (predict price/demand for new areas,
 │   & Support Decisions│    support housing project planning)
 └──────────────────────┘

**Explanation:**
1. **Collect data** — gather area-level housing and demographic data.
2. **Clean & prepare** — remove/handle missing values and non-numeric columns (like `Address`) so the data is model-ready.
3. **Split data** — separate data into a training set (to teach the model) and a test set (to check how well it learned).
4. **Train a model** — start simple, e.g. Linear Regression, to learn the relationship between features and price.
5. **Evaluate** — measure how close predictions are to actual prices using error metrics (e.g. RMSE).
6. **Apply the model** — use it to predict prices/demand for new areas and support housing project planning.
